# 16 — Dashboard metrics & Planner context

**Research question:** Có thể xuất facts có nguồn rõ ràng và chỉ đúng subject mà không tự tạo kế hoạch từ dữ liệu chưa xác minh không?

Baseline: deterministic context builder + versioned JSON Schemas. Dùng cùng fixture synthetic, không gọi LLM, không sửa mục tiêu.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
config = json.loads((ROOT / "configs/progress_phase4.json").read_text(encoding="utf-8"))
progress = json.loads((ROOT / "experiments/body_progress_phase4/progress.json").read_text(encoding="utf-8"))


In [2]:
from jsonschema import Draft202012Validator, FormatChecker
dashboard = json.loads((ROOT / "experiments/body_progress_phase4/dashboard.json").read_text())
context = json.loads((ROOT / "experiments/body_progress_phase4/planner_context.json").read_text())
selected_context = json.loads((ROOT / "experiments/body_progress_phase4/planner_context_selected_demo.json").read_text())
for name, payload in [("body_progress_dashboard", dashboard), ("body_progress_planner_context", context), ("body_progress_planner_context", selected_context)]:
    schema = json.loads((ROOT / "schemas" / (name + ".schema.json")).read_text())
    Draft202012Validator(schema, format_checker=FormatChecker()).validate(payload)
assert "audit_rows" not in dashboard
assert all(s["subject_id"] == dashboard["subject_id"] for s in dashboard["series"])
print("Subject-scoped schemas passed")

Subject-scoped schemas passed


In [3]:
print("Unselected source conflicts:")
display(context["needs_source_selection"])
print("Fixture's explicit source selection (not an automatic preference rule):")
display([{key: fact[key] for key in ["field", "provenance", "source", "measurement_method", "device_id", "verification_band"]} for fact in selected_context["facts"]])
assert all(fact["provenance"] in ["observed", "extracted"] and fact["verification_band"] == "reviewed" for fact in selected_context["facts"])
assert not selected_context["plan_generation_authorized"]
assert selected_context["requires_safety_screening"] and selected_context["goal_changes_require_user_approval"]
display(selected_context["interpretation_limits"])

Unselected source conflicts:


[{'field': 'percent_body_fat',
  'candidate_stream_ids': ['47d9310d5645d98cfd20', 'bbdc3473f0c7cbc03cb6']},
 {'field': 'weight_kg',
  'candidate_stream_ids': ['1611e440fca57cc32044', '2f81218cbea8f485d78d']}]

Fixture's explicit source selection (not an automatic preference rule):


[{'field': 'percent_body_fat',
  'provenance': 'observed',
  'source': 'manual_measurement',
  'measurement_method': 'bioimpedance_device',
  'device_id': 'BIA-A',
  'verification_band': 'reviewed'},
 {'field': 'skeletal_muscle_mass_kg',
  'provenance': 'observed',
  'source': 'manual_measurement',
  'measurement_method': 'bioimpedance_device',
  'device_id': 'BIA-A',
  'verification_band': 'reviewed'},
 {'field': 'waist_cm',
  'provenance': 'observed',
  'source': 'manual_measurement',
  'measurement_method': 'tape_measure',
  'device_id': 'tape-A',
  'verification_band': 'reviewed'},
 {'field': 'weight_kg',
  'provenance': 'observed',
  'source': 'manual_measurement',
  'measurement_method': 'digital_scale',
  'device_id': 'scale-A',
  'verification_band': 'reviewed'}]

'Reviewed describes entry/transcription review, not clinical validation. BIA results are device estimates; no causal or clinical conclusion.'

## Integration boundary / error analysis / conclusion

Adapter Phase 1 giữ `extracted` sau human review, không đổi thành observed; không có ngày đo thì giữ null. Adapter Phase 3 cần capture timestamp do caller cung cấp, giữ `inferred`, checkpoint version, interval evidence và quality/OOD flags. Uncertainty deployment gate Phase 3 chưa qua nên không dùng ML vào planner facts.

Có JSON/schema và CLI research, chưa có API/database/UI chạy trên người dùng. Consent ID chỉ là tham chiếu; server Phase 5 phải xác thực người dùng, quyền dữ liệu, consent store và safety screening độc lập LLM. Không có mục tiêu hay kế hoạch tự động thay đổi. Phase 4 hoàn thành thử phần mềm bằng synthetic; cần nguồn lịch sử đo có quyền sử dụng để kiểm thực tế.